
# Multi-Agent Supervisor Pattern

**Day 3 — RAG & Agents · Practical 6 of 6 · Companion to the "Advanced Agents & Protocols"
deck**

> **Running in Google Colab:** works on the default **CPU runtime** — LLM API calls only.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build a supervisor agent that routes tasks to specialist worker agents
2. Compose multiple single-agent graphs (like Notebook 5's) into one coordinated system
3. Add a Human-in-the-Loop interrupt gating a high-risk final step

## Why This Matters for a Law Firm

The deck names the Supervisor pattern as the recommended default for a first real multi-agent
system — widest framework support, best-understood failure modes. This notebook builds exactly
that: one supervisor, two specialists, and a human approval gate before the riskiest action.

## Notebook Workflow

```mermaid
flowchart TD
    A["Incoming task"] --> S["Supervisor agent\n(decides routing)"]
    S -->|"clause research"| W1["Clause Research\nWorker"]
    S -->|"risk analysis"| W2["Risk Analysis\nWorker"]
    W1 --> S
    W2 --> S
    S -->|"task complete"| H["Human-in-the-Loop\ncheckpoint"]
    H -->|"approved"| End(["Final output"])



## Section 1 — Setup


In [ ]:

%pip install -q langgraph langchain-openai

import os

def get_api_key(env_var_name):
    try:
        from google.colab import userdata
        key = userdata.get(env_var_name)
        if key:
            return key
    except ImportError:
        pass
    return os.environ.get(env_var_name)

OPENAI_API_KEY = get_api_key("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Add an OPENAI_API_KEY secret in Colab (key icon, left sidebar).")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key configured.")



## Section 2 — Define the Shared State

The supervisor and both workers all read from and write to one shared state object -- the same
State concept from Notebook 5, now shared across multiple agents instead of just one.


In [ ]:

from typing import Annotated, TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

class SupervisorState(TypedDict):
    task: str
    clause_findings: str
    risk_findings: str
    next_worker: str
    final_output: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("LLM configured.")



## Section 3 — Define the Two Worker Agents

Each worker is a focused specialist -- one researches relevant clauses, the other assesses
risk. Neither worker needs to know anything about the other's job.


In [ ]:

def clause_research_worker(state: SupervisorState) -> SupervisorState:
    response = llm.invoke([
        SystemMessage(content="You are a legal research specialist. Identify which standard contract clauses are relevant to the task, briefly."),
        HumanMessage(content=state["task"]),
    ])
    return {**state, "clause_findings": response.content}


def risk_analysis_worker(state: SupervisorState) -> SupervisorState:
    response = llm.invoke([
        SystemMessage(content="You are a contract risk analyst. Identify the top risk factors relevant to the task, briefly."),
        HumanMessage(content=state["task"]),
    ])
    return {**state, "risk_findings": response.content}

print("Worker agents defined: clause_research_worker, risk_analysis_worker")



## Section 4 — Define the Supervisor

The supervisor decides which worker(s) to call, and when the task is complete. It's the
routing brain — this maps directly to the deck's "supervisor coordinates specialist workers,
making ongoing decisions about routing" description.


In [ ]:

def supervisor_node(state: SupervisorState) -> SupervisorState:
    # Decide what still needs to happen based on what's already been done.
    if not state.get("clause_findings"):
        return {**state, "next_worker": "clause_research"}
    if not state.get("risk_findings"):
        return {**state, "next_worker": "risk_analysis"}
    return {**state, "next_worker": "done"}


def route_from_supervisor(state: SupervisorState) -> Literal["clause_research", "risk_analysis", "synthesize"]:
    if state["next_worker"] == "clause_research":
        return "clause_research"
    elif state["next_worker"] == "risk_analysis":
        return "risk_analysis"
    else:
        return "synthesize"


def synthesize_node(state: SupervisorState) -> SupervisorState:
    # Combine both workers' outputs into one final answer.
    response = llm.invoke([
        SystemMessage(content="Combine the following findings into one concise summary for a legal associate."),
        HumanMessage(content=f"Clause findings:\n{state['clause_findings']}\n\nRisk findings:\n{state['risk_findings']}"),
    ])
    return {**state, "final_output": response.content}

print("Supervisor logic defined.")



## Section 5 — Build the Graph

Supervisor routes to whichever worker hasn't run yet; each worker returns to the supervisor;
once both workers are done, the supervisor routes to synthesis.


In [ ]:

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("clause_research", clause_research_worker)
graph.add_node("risk_analysis", risk_analysis_worker)
graph.add_node("synthesize", synthesize_node)

graph.set_entry_point("supervisor")
graph.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {"clause_research": "clause_research", "risk_analysis": "risk_analysis", "synthesize": "synthesize"},
)
graph.add_edge("clause_research", "supervisor")
graph.add_edge("risk_analysis", "supervisor")
graph.add_edge("synthesize", END)

app = graph.compile()
print("Multi-agent graph compiled: supervisor + 2 workers + synthesis.")



## Section 6 — Run the Multi-Agent System

Watch the supervisor route to both workers in turn, then synthesize their findings.


In [ ]:

initial_state = {
    "task": "We're drafting a new vendor services agreement with a data-processing vendor that will handle client PII.",
    "clause_findings": "",
    "risk_findings": "",
    "next_worker": "",
    "final_output": "",
}

result = app.invoke(initial_state)

print("CLAUSE RESEARCH WORKER OUTPUT:\n")
print(result["clause_findings"])
print("\n" + "=" * 70 + "\n")
print("RISK ANALYSIS WORKER OUTPUT:\n")
print(result["risk_findings"])
print("\n" + "=" * 70 + "\n")
print("SYNTHESIZED FINAL OUTPUT:\n")
print(result["final_output"])



## Section 7 — Adding a Human-in-the-Loop Checkpoint

Per the deck, HITL inserts an explicit approval checkpoint before a risky step. Here, we gate
the transition from "synthesized findings" to "final approved output" -- a human must review
before the summary is treated as final. LangGraph implements this with an `interrupt_before`
compile option.


In [ ]:

from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
app_with_hitl = graph.compile(checkpointer=checkpointer, interrupt_before=["synthesize"])

config = {"configurable": {"thread_id": "demo-thread-1"}}

# Run until the interrupt point (right before "synthesize")
interrupted_state = app_with_hitl.invoke(initial_state, config=config)

print("Execution paused BEFORE synthesis, for human review.")
print("\nClause findings so far:\n", interrupted_state["clause_findings"][:200], "...")
print("\nRisk findings so far:\n", interrupted_state["risk_findings"][:200], "...")



## Section 8 — Approve and Resume

In a real system, a human would review the state above and either approve, edit, or reject it.
Here we simulate approval by simply resuming execution -- `None` as input tells LangGraph to
continue from exactly where it paused, using the graph's checkpointed state.


In [ ]:

final_result = app_with_hitl.invoke(None, config=config)

print("APPROVED -- execution resumed and completed.\n")
print("Final synthesized output:\n")
print(final_result["final_output"])



## Key Takeaways

1. **The supervisor pattern composes cleanly** -- each worker is a simple, focused function;
   the supervisor's ONLY job is deciding what happens next, based on shared state.
2. **This is the same graph-and-state mechanism as Notebook 5's single agent** -- multi-agent
   systems in LangGraph aren't a different framework, they're the same primitives (nodes, edges,
   conditional routing, shared state) applied to more nodes.
3. **The Human-in-the-Loop interrupt** genuinely pauses execution and preserves state -- Section
   7's `interrupted_state` shows exactly what a reviewer would see before approving, and Section
   8 shows execution resuming from precisely that point, not restarting from scratch.

**This completes Day 3's hands-on practicals.** From vanilla RAG through hybrid retrieval,
self-correction, multimodal retrieval, single agents, and now multi-agent systems -- the full
RAG-and-agents stack from the decks is now runnable code. **Next up:** Day 4 — AI Security &
Legal Compliance.
